# Advanced JSON Schema — Tutorial-Style Problems with Complete Solutions

This notebook continues the topic of **JSON Schema validation in Python**, but it is deliberately structured like a guided tutorial rather than a compact reference.

Instead of jumping directly to finished schemas, every major problem is broken into small steps:

1. understand the shape of the JSON document,
2. identify which fields are required,
3. constrain primitive values,
4. introduce nested structures,
5. add cross-field rules,
6. validate good and bad examples,
7. inspect validation errors,
8. refine the schema,
9. summarize the final design.

The goal is not only to *see* a correct schema, but to learn how to **build one incrementally**.


## What We Will Practice

The problems in this notebook cover:

- strict object schemas,
- required vs optional vs nullable fields,
- numbers and numeric ranges,
- string lengths and regular expressions,
- enums and constants,
- nested objects,
- arrays,
- reusable definitions with `$defs`,
- references with `$ref`,
- `oneOf`,
- `anyOf`,
- `allOf`,
- `not`,
- `if` / `then` / `else`,
- `dependentRequired`,
- `contains`,
- `minContains`,
- `prefixItems`,
- `propertyNames`,
- `patternProperties`,
- recursive schemas,
- versioned API documents,
- useful validation error reporting,
- schema testing,
- and business rules that are better handled in Python than in JSON Schema.


## Setup

We will use the `jsonschema` package.

If necessary, uncomment the following command in a Jupyter environment:

```python
%pip install -U jsonschema
```

This notebook targets **JSON Schema Draft 2020-12**.


In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from pprint import pprint

from jsonschema import Draft202012Validator, FormatChecker
from jsonschema.exceptions import ValidationError, SchemaError

FORMAT_CHECKER = FormatChecker()

print("Setup complete.")

Setup complete.


Before solving the larger problems, we will create a few helper functions.

A schema validator can stop at the first error, but during development it is usually much more useful to see **all** validation errors.


In [2]:
def path_to_string(path):
    result = "$"

    for part in path:
        if isinstance(part, int):
            result += f"[{part}]"
        else:
            result += f".{part}"

    return result


def collect_errors(instance, schema, *, check_formats=True):
    validator = Draft202012Validator(
        schema,
        format_checker=FORMAT_CHECKER if check_formats else None,
    )

    errors = list(validator.iter_errors(instance))

    return sorted(
        errors,
        key=lambda error: (
            list(error.absolute_path),
            error.validator or "",
            error.message,
        ),
    )


def print_errors(instance, schema, *, check_formats=True):
    errors = collect_errors(
        instance,
        schema,
        check_formats=check_formats,
    )

    if not errors:
        print("VALID")
        return

    print(f"INVALID: {len(errors)} error(s)")

    for index, error in enumerate(errors, start=1):
        path = path_to_string(error.absolute_path)
        print(
            f"{index:>2}. {path} "
            f"[{error.validator}] "
            f"{error.message}"
        )


def is_valid(instance, schema):
    return not collect_errors(instance, schema)

A useful habit is to validate the **schema itself** before trusting it.

The following helper asks the Draft 2020-12 validator to check whether the schema definition is valid.


In [3]:
def check_schema(schema):
    Draft202012Validator.check_schema(schema)
    print("Schema definition is valid.")

---

# Problem 1 — Building a Strict Student Record

Suppose we are building an API for a university.

A student record should eventually look like this:

```json
{
    "studentId": "STU-20260001",
    "firstName": "Ada",
    "lastName": "Lovelace",
    "year": 2,
    "gpa": 3.92,
    "status": "active"
}
```

We will build the schema gradually.


## Step 1 — Start with the Outer Shape

The document must be a JSON object.

At this stage, we will not constrain any fields yet.


In [4]:
student_schema_step_1 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object"
}

check_schema(student_schema_step_1)

Schema definition is valid.


This schema accepts **any object**.

That means all of these are currently valid:

```python
{}
{"x": 1}
{"studentId": None}
```

That is expected. We have only described the outermost type.


In [5]:
student_step_1_examples = [
    {},
    {"x": 1},
    {"studentId": None},
]

for example in student_step_1_examples:
    print(example, "->", is_valid(example, student_schema_step_1))

{} -> True
{'x': 1} -> True
{'studentId': None} -> True


## Step 2 — Describe the Properties

Now we can define the types of the known fields.

Notice that defining a field inside `properties` does **not** automatically make it required.


In [6]:
student_schema_step_2 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "studentId": {"type": "string"},
        "firstName": {"type": "string"},
        "lastName": {"type": "string"},
        "year": {"type": "integer"},
        "gpa": {"type": "number"},
        "status": {"type": "string"},
    },
}

check_schema(student_schema_step_2)

Schema definition is valid.


This means an empty object is still valid because we have not declared any required properties.


In [7]:
print_errors({}, student_schema_step_2)

VALID


## Step 3 — Add Required Fields

For our API, every student must have:

- `studentId`
- `firstName`
- `lastName`
- `year`
- `status`

The GPA will remain optional.


In [8]:
student_schema_step_3 = {
    **student_schema_step_2,
    "required": [
        "studentId",
        "firstName",
        "lastName",
        "year",
        "status",
    ],
}

print_errors({}, student_schema_step_3)

INVALID: 5 error(s)
 1. $ [required] 'firstName' is a required property
 2. $ [required] 'lastName' is a required property
 3. $ [required] 'status' is a required property
 4. $ [required] 'studentId' is a required property
 5. $ [required] 'year' is a required property


The validator now reports missing required properties.

This is an important distinction:

- `properties` tells us **what a property must look like if it exists**,
- `required` tells us **which properties must exist**.


## Step 4 — Add Stronger Constraints

Now let us make the schema more precise.

Requirements:

- `studentId` must use the format `STU-` followed by exactly 8 digits,
- names must not be empty,
- `year` must be from 1 to 6,
- `gpa` must be from 0.0 to 4.0,
- `status` must be one of `active`, `suspended`, or `graduated`.


In [9]:
student_schema_step_4 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "studentId": {
            "type": "string",
            "pattern": "^STU-[0-9]{8}$",
        },
        "firstName": {
            "type": "string",
            "minLength": 1,
        },
        "lastName": {
            "type": "string",
            "minLength": 1,
        },
        "year": {
            "type": "integer",
            "minimum": 1,
            "maximum": 6,
        },
        "gpa": {
            "type": "number",
            "minimum": 0,
            "maximum": 4,
        },
        "status": {
            "type": "string",
            "enum": ["active", "suspended", "graduated"],
        },
    },
    "required": [
        "studentId",
        "firstName",
        "lastName",
        "year",
        "status",
    ],
}

check_schema(student_schema_step_4)

Schema definition is valid.


Let us test a valid example.


In [10]:
student_1 = {
    "studentId": "STU-20260001",
    "firstName": "Ada",
    "lastName": "Lovelace",
    "year": 2,
    "gpa": 3.92,
    "status": "active",
}

print_errors(student_1, student_schema_step_4)

VALID


Now let us introduce several independent mistakes at once.


In [11]:
student_2 = {
    "studentId": "20260001",
    "firstName": "",
    "lastName": "Lovelace",
    "year": 8,
    "gpa": 4.7,
    "status": "pending",
}

print_errors(student_2, student_schema_step_4)

INVALID: 5 error(s)
 1. $.firstName [minLength] '' should be non-empty
 2. $.gpa [maximum] 4.7 is greater than the maximum of 4
 3. $.status [enum] 'pending' is not one of ['active', 'suspended', 'graduated']
 4. $.studentId [pattern] '20260001' does not match '^STU-[0-9]{8}$'
 5. $.year [maximum] 8 is greater than the maximum of 6


## Step 5 — Reject Unknown Properties

So far, this record is still accepted:

```json
{
    "studentId": "STU-20260001",
    "firstName": "Ada",
    "lastName": "Lovelace",
    "year": 2,
    "status": "active",
    "debug": true
}
```

For a strict API contract, we may want to reject fields we did not define.


In [12]:
STUDENT_SCHEMA = {
    **student_schema_step_4,
    "additionalProperties": False,
}

check_schema(STUDENT_SCHEMA)

Schema definition is valid.


In [13]:
student_with_extra_field = {
    "studentId": "STU-20260001",
    "firstName": "Ada",
    "lastName": "Lovelace",
    "year": 2,
    "status": "active",
    "debug": True,
}

print_errors(student_with_extra_field, STUDENT_SCHEMA)

INVALID: 1 error(s)
 1. $ [additionalProperties] Additional properties are not allowed ('debug' was unexpected)


## Problem 1 Summary

We built the schema incrementally:

1. object type,
2. property definitions,
3. required fields,
4. detailed constraints,
5. strict rejection of unknown fields.

This is a useful general workflow for schema design.


---

# Problem 2 — Optional Is Not the Same as Nullable

This distinction causes many real API bugs.

Suppose a user profile has:

- required `username`,
- optional `nickname`,
- required `middleName`, but the value may be `null`.

We want to understand the difference carefully.


## Step 1 — Optional Property

If `nickname` is defined in `properties` but is not listed in `required`, then the field may be absent.

However, if the field is present, it must satisfy its schema.


In [14]:
optional_nickname_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "nickname": {
            "type": "string",
            "minLength": 1,
        }
    },
    "additionalProperties": False,
}

examples = [
    {},
    {"nickname": "Ace"},
    {"nickname": None},
]

for example in examples:
    print(example)
    print_errors(example, optional_nickname_schema)
    print()

{}
VALID

{'nickname': 'Ace'}
VALID

{'nickname': None}
INVALID: 1 error(s)
 1. $.nickname [type] None is not of type 'string'



The third case fails because `null` is not a string.

So **optional** means:

> the property does not need to exist.

It does not mean:

> the property may contain `null`.


## Step 2 — Nullable Property

To allow a JSON `null`, we can allow more than one JSON type.


In [15]:
nullable_middle_name_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "middleName": {
            "type": ["string", "null"]
        }
    },
    "required": ["middleName"],
    "additionalProperties": False,
}

examples = [
    {"middleName": "Marie"},
    {"middleName": None},
    {},
]

for example in examples:
    print(example)
    print_errors(example, nullable_middle_name_schema)
    print()

{'middleName': 'Marie'}
VALID

{'middleName': None}
VALID

{}
INVALID: 1 error(s)
 1. $ [required] 'middleName' is a required property



The empty object fails because `middleName` is required.

But the explicit JSON value `null` is allowed.


## Step 3 — Final Profile Schema


In [16]:
PROFILE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "username": {
            "type": "string",
            "minLength": 3,
        },
        "nickname": {
            "type": "string",
            "minLength": 1,
        },
        "middleName": {
            "type": ["string", "null"],
        },
    },
    "required": ["username", "middleName"],
    "additionalProperties": False,
}

profile_examples = [
    {
        "username": "alice",
        "middleName": None,
    },
    {
        "username": "bob",
        "nickname": "B",
        "middleName": "James",
    },
    {
        "username": "carol",
        "nickname": None,
        "middleName": None,
    },
]

for example in profile_examples:
    pprint(example)
    print_errors(example, PROFILE_SCHEMA)
    print("-" * 60)

{'middleName': None, 'username': 'alice'}
VALID
------------------------------------------------------------
{'middleName': 'James', 'nickname': 'B', 'username': 'bob'}
VALID
------------------------------------------------------------
{'middleName': None, 'nickname': None, 'username': 'carol'}
INVALID: 1 error(s)
 1. $.nickname [type] None is not of type 'string'
------------------------------------------------------------


## Problem 2 Summary

Remember:

- **required** controls whether a key must be present,
- **type** controls which values are allowed once the key is present.

Presence and nullability are separate design decisions.


---

# Problem 3 — Reusable Nested Objects with `$defs` and `$ref`

Suppose we are building an invoice API.

An invoice contains:

- a seller address,
- a billing address,
- optionally a shipping address.

All three addresses follow the same structure.

Copying the address schema three times would work, but it would create duplication.

A better solution is to define the address once and reuse it.


## Step 1 — Design the Address Object

An address requires:

- `line1`,
- `city`,
- `postalCode`,
- `countryCode`.

`line2` is optional.

We will use a two-letter uppercase country code.


In [17]:
address_schema = {
    "type": "object",
    "properties": {
        "line1": {
            "type": "string",
            "minLength": 1,
        },
        "line2": {
            "type": "string",
        },
        "city": {
            "type": "string",
            "minLength": 1,
        },
        "postalCode": {
            "type": "string",
            "minLength": 1,
        },
        "countryCode": {
            "type": "string",
            "pattern": "^[A-Z]{2}$",
        },
    },
    "required": [
        "line1",
        "city",
        "postalCode",
        "countryCode",
    ],
    "additionalProperties": False,
}

check_schema({
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    **address_schema,
})

Schema definition is valid.


## Step 2 — Place the Definition in `$defs`

`$defs` is a container for reusable schema fragments.


In [18]:
invoice_schema_step_2 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "address": address_schema,
    },
    "type": "object",
}

check_schema(invoice_schema_step_2)

Schema definition is valid.


## Step 3 — Reference the Address

A local reference looks like:

```json
{"$ref": "#/$defs/address"}
```

The `#` means we are referencing something inside the current schema document.


In [19]:
INVOICE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "address": address_schema,
    },
    "type": "object",
    "properties": {
        "invoiceId": {
            "type": "string",
            "pattern": "^INV-[0-9]{6}$",
        },
        "sellerAddress": {
            "$ref": "#/$defs/address"
        },
        "billingAddress": {
            "$ref": "#/$defs/address"
        },
        "shippingAddress": {
            "$ref": "#/$defs/address"
        },
    },
    "required": [
        "invoiceId",
        "sellerAddress",
        "billingAddress",
    ],
    "additionalProperties": False,
}

check_schema(INVOICE_SCHEMA)

Schema definition is valid.


## Step 4 — Validate a Good Invoice


In [20]:
invoice_1 = {
    "invoiceId": "INV-123456",
    "sellerAddress": {
        "line1": "10 Main Street",
        "city": "London",
        "postalCode": "SW1A 1AA",
        "countryCode": "GB",
    },
    "billingAddress": {
        "line1": "42 Example Road",
        "city": "Sofia",
        "postalCode": "1000",
        "countryCode": "BG",
    },
}

print_errors(invoice_1, INVOICE_SCHEMA)

VALID


## Step 5 — A Broken Nested Address

The validator reports the path all the way into the nested object.


In [21]:
invoice_2 = {
    "invoiceId": "INV-123456",
    "sellerAddress": {
        "line1": "",
        "city": "London",
        "postalCode": "SW1A 1AA",
        "countryCode": "gb",
    },
    "billingAddress": {
        "line1": "42 Example Road",
        "city": "Sofia",
        "postalCode": "1000",
        "countryCode": "BG",
        "unexpected": True,
    },
}

print_errors(invoice_2, INVOICE_SCHEMA)

INVALID: 3 error(s)
 1. $.billingAddress [additionalProperties] Additional properties are not allowed ('unexpected' was unexpected)
 2. $.sellerAddress.countryCode [pattern] 'gb' does not match '^[A-Z]{2}$'
 3. $.sellerAddress.line1 [minLength] '' should be non-empty


## Problem 3 Summary

Use `$defs` and `$ref` when:

- the same structure appears more than once,
- you want to avoid copy/paste schemas,
- you want one source of truth for shared structures.


---

# Problem 4 — Arrays of Objects

We will extend the invoice idea with line items.

Each line item contains:

- `sku`,
- `description`,
- `quantity`,
- `unitPrice`.

The invoice must contain at least one line item.


## Step 1 — Build the Line Item Schema


In [22]:
line_item_schema = {
    "type": "object",
    "properties": {
        "sku": {
            "type": "string",
            "pattern": "^[A-Z0-9-]{3,30}$",
        },
        "description": {
            "type": "string",
            "minLength": 1,
        },
        "quantity": {
            "type": "integer",
            "minimum": 1,
        },
        "unitPrice": {
            "type": "number",
            "minimum": 0,
        },
    },
    "required": [
        "sku",
        "description",
        "quantity",
        "unitPrice",
    ],
    "additionalProperties": False,
}

## Step 2 — Define the Array

The `items` keyword controls the schema of each array element.

The `minItems` keyword ensures the array is not empty.


In [23]:
invoice_with_items_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "lineItem": line_item_schema,
    },
    "type": "object",
    "properties": {
        "invoiceId": {
            "type": "string",
            "pattern": "^INV-[0-9]{6}$",
        },
        "items": {
            "type": "array",
            "minItems": 1,
            "items": {
                "$ref": "#/$defs/lineItem"
            },
        },
    },
    "required": [
        "invoiceId",
        "items",
    ],
    "additionalProperties": False,
}

check_schema(invoice_with_items_schema)

Schema definition is valid.


## Step 3 — Validate Multiple Array Elements

A useful point to observe is the error path.

If the second item is broken, we should see a path such as:

```text
$.items[1].quantity
```


In [24]:
invoice_items_example = {
    "invoiceId": "INV-000777",
    "items": [
        {
            "sku": "BOOK-1",
            "description": "Python Book",
            "quantity": 2,
            "unitPrice": 39.95,
        },
        {
            "sku": "bad sku",
            "description": "",
            "quantity": 0,
            "unitPrice": -5,
        },
    ],
}

print_errors(invoice_items_example, invoice_with_items_schema)

INVALID: 4 error(s)
 1. $.items[1].description [minLength] '' should be non-empty
 2. $.items[1].quantity [minimum] 0 is less than the minimum of 1
 3. $.items[1].sku [pattern] 'bad sku' does not match '^[A-Z0-9-]{3,30}$'
 4. $.items[1].unitPrice [minimum] -5 is less than the minimum of 0


## Step 4 — Unique Arrays

Suppose an invoice also has tags.

If tags must not repeat, `uniqueItems` can help.


In [25]:
tags_schema = {
    "type": "array",
    "uniqueItems": True,
    "items": {
        "type": "string",
        "minLength": 1,
    },
}

for tags in [
    ["priority", "international"],
    ["priority", "priority"],
]:
    print(tags)
    print_errors(tags, tags_schema)
    print()

['priority', 'international']
VALID

['priority', 'priority']
INVALID: 1 error(s)
 1. $ [uniqueItems] ['priority', 'priority'] has non-unique elements



A limitation is important here:

`uniqueItems` compares entire array values.

It cannot directly express:

> each line item must have a unique `sku`.

That is a cross-item business rule and often belongs in application-level code.


---

# Problem 5 — Exactly One Variant with `oneOf`

Suppose a login request supports three authentication methods:

1. password,
2. API key,
3. OAuth token.

The request must match **exactly one** method.

This is a classic use case for `oneOf`.


## Step 1 — Why Not One Giant Object?

We could create one object with all possible fields:

- `username`
- `password`
- `apiKey`
- `provider`
- `accessToken`

But then we would have to express complicated rules about which combinations are allowed.

A cleaner design is to model each authentication method separately.


## Step 2 — Password Branch


In [26]:
password_auth_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "password"
        },
        "username": {
            "type": "string",
            "minLength": 1,
        },
        "password": {
            "type": "string",
            "minLength": 8,
        },
    },
    "required": [
        "type",
        "username",
        "password",
    ],
    "additionalProperties": False,
}

The `const` keyword is especially useful as a discriminator.

It makes the branch clearly identifiable.


## Step 3 — API Key and OAuth Branches


In [27]:
api_key_auth_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "api_key"
        },
        "apiKey": {
            "type": "string",
            "minLength": 16,
        },
    },
    "required": [
        "type",
        "apiKey",
    ],
    "additionalProperties": False,
}


oauth_auth_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "oauth"
        },
        "provider": {
            "enum": ["google", "github", "microsoft"],
        },
        "accessToken": {
            "type": "string",
            "minLength": 10,
        },
    },
    "required": [
        "type",
        "provider",
        "accessToken",
    ],
    "additionalProperties": False,
}

## Step 4 — Combine Them with `oneOf`


In [28]:
LOGIN_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "oneOf": [
        password_auth_schema,
        api_key_auth_schema,
        oauth_auth_schema,
    ],
}

check_schema(LOGIN_SCHEMA)

Schema definition is valid.


## Step 5 — Test Valid Requests


In [29]:
login_examples = [
    {
        "type": "password",
        "username": "alice",
        "password": "correct-horse-battery",
    },
    {
        "type": "api_key",
        "apiKey": "0123456789abcdef",
    },
    {
        "type": "oauth",
        "provider": "github",
        "accessToken": "token-1234567890",
    },
]

for example in login_examples:
    pprint(example)
    print_errors(example, LOGIN_SCHEMA)
    print("-" * 60)

{'password': 'correct-horse-battery', 'type': 'password', 'username': 'alice'}
VALID
------------------------------------------------------------
{'apiKey': '0123456789abcdef', 'type': 'api_key'}
VALID
------------------------------------------------------------
{'accessToken': 'token-1234567890', 'provider': 'github', 'type': 'oauth'}
VALID
------------------------------------------------------------


## Step 6 — Test a Mixed Request

This request says it is password authentication but contains an API key instead.


In [30]:
mixed_login = {
    "type": "password",
    "username": "alice",
    "apiKey": "0123456789abcdef",
}

print_errors(mixed_login, LOGIN_SCHEMA)

INVALID: 1 error(s)
 1. $ [oneOf] {'type': 'password', 'username': 'alice', 'apiKey': '0123456789abcdef'} is not valid under any of the given schemas


## Step 7 — Inspect Branch Errors

A `oneOf` failure often contains nested errors from the attempted alternatives.

Those nested errors are available through `ValidationError.context`.


In [31]:
errors = collect_errors(mixed_login, LOGIN_SCHEMA)

for error in errors:
    print("TOP LEVEL:")
    print(error.message)

    print("\nBRANCH DETAILS:")
    for suberror in error.context:
        print(
            " -",
            path_to_string(suberror.absolute_path),
            suberror.message,
        )

TOP LEVEL:
{'type': 'password', 'username': 'alice', 'apiKey': '0123456789abcdef'} is not valid under any of the given schemas

BRANCH DETAILS:
 - $ 'password' is a required property
 - $ Additional properties are not allowed ('apiKey' was unexpected)
 - $.type 'api_key' was expected
 - $ Additional properties are not allowed ('username' was unexpected)
 - $.type 'oauth' was expected
 - $ 'provider' is a required property
 - $ 'accessToken' is a required property
 - $ Additional properties are not allowed ('apiKey', 'username' were unexpected)


## Problem 5 Summary

Use `oneOf` when the data must match **exactly one** alternative.

A `const` discriminator such as `"type": "oauth"` makes the alternatives easier to understand, validate, and maintain.


---

# Problem 6 — At Least One Choice with `anyOf`

Now consider a notification destination.

A user must provide at least one contact method:

- `email`,
- `phone`,
- `deviceToken`.

Unlike the previous problem, providing more than one is allowed.

That means `anyOf` is a better fit than `oneOf`.


## Step 1 — Base Properties


In [32]:
notification_target_base = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "email": {
            "type": "string",
            "format": "email",
        },
        "phone": {
            "type": "string",
            "pattern": "^\\+[1-9][0-9]{7,14}$",
        },
        "deviceToken": {
            "type": "string",
            "minLength": 20,
        },
    },
    "additionalProperties": False,
}

## Step 2 — Require At Least One

A useful JSON Schema pattern is:

```json
"anyOf": [
    {"required": ["email"]},
    {"required": ["phone"]},
    {"required": ["deviceToken"]}
]
```

Each branch simply asks whether a specific field is present.


In [33]:
NOTIFICATION_TARGET_SCHEMA = {
    **notification_target_base,
    "anyOf": [
        {"required": ["email"]},
        {"required": ["phone"]},
        {"required": ["deviceToken"]},
    ],
}

check_schema(NOTIFICATION_TARGET_SCHEMA)

Schema definition is valid.


In [34]:
targets = [
    {"email": "person@example.com"},
    {"phone": "+359888123456"},
    {
        "email": "person@example.com",
        "phone": "+359888123456",
    },
    {},
]

for target in targets:
    print(target)
    print_errors(target, NOTIFICATION_TARGET_SCHEMA)
    print()

{'email': 'person@example.com'}
VALID

{'phone': '+359888123456'}
VALID

{'email': 'person@example.com', 'phone': '+359888123456'}
VALID

{}
INVALID: 1 error(s)
 1. $ [anyOf] {} is not valid under any of the given schemas



## Problem 6 Summary

- `oneOf` means exactly one branch must match.
- `anyOf` means one or more branches may match.

Choosing between them is a semantic modeling decision.


---

# Problem 7 — Conditional Rules with `if` / `then` / `else`

Suppose we have a delivery request.

The field `deliveryType` is either:

- `home`,
- `pickup`.

Rules:

- home delivery requires an `address`,
- pickup requires a `storeId`.

We will express this with a conditional schema.


## Step 1 — Define All Possible Properties

A practical pattern is to define the possible property names at the outer level.

This makes it easier to keep the object strict with `additionalProperties: false`.


In [35]:
delivery_schema_step_1 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "deliveryType": {
            "enum": ["home", "pickup"],
        },
        "address": {
            "type": "string",
            "minLength": 1,
        },
        "storeId": {
            "type": "string",
            "pattern": "^STORE-[0-9]{4}$",
        },
    },
    "required": ["deliveryType"],
    "additionalProperties": False,
}

At this point, the schema knows the allowed fields, but it does not yet know which fields belong to which delivery type.


In [36]:
for example in [
    {"deliveryType": "home"},
    {"deliveryType": "pickup"},
]:
    print(example)
    print_errors(example, delivery_schema_step_1)
    print()

{'deliveryType': 'home'}
VALID

{'deliveryType': 'pickup'}
VALID



## Step 2 — Add the Condition

The condition checks whether:

```json
"deliveryType": "home"
```


In [37]:
home_condition = {
    "properties": {
        "deliveryType": {
            "const": "home"
        }
    },
    "required": ["deliveryType"],
}

The explicit `required` inside the condition is a good defensive habit.

Without it, a property-only condition can behave in ways that surprise beginners when the property is absent.


## Step 3 — Add `then` and `else`

If the request is for home delivery:

- `address` is required,
- `storeId` must not be present.

Otherwise:

- `storeId` is required,
- `address` must not be present.


In [38]:
DELIVERY_SCHEMA = {
    **delivery_schema_step_1,
    "if": home_condition,
    "then": {
        "required": ["address"],
        "not": {
            "required": ["storeId"]
        },
    },
    "else": {
        "required": ["storeId"],
        "not": {
            "required": ["address"]
        },
    },
}

check_schema(DELIVERY_SCHEMA)

Schema definition is valid.


## Step 4 — Validate the Variants


In [39]:
delivery_cases = [
    {
        "deliveryType": "home",
        "address": "10 Main Street",
    },
    {
        "deliveryType": "pickup",
        "storeId": "STORE-0042",
    },
    {
        "deliveryType": "home",
        "storeId": "STORE-0042",
    },
    {
        "deliveryType": "pickup",
        "address": "10 Main Street",
    },
]

for case in delivery_cases:
    pprint(case)
    print_errors(case, DELIVERY_SCHEMA)
    print("-" * 60)

{'address': '10 Main Street', 'deliveryType': 'home'}
VALID
------------------------------------------------------------
{'deliveryType': 'pickup', 'storeId': 'STORE-0042'}
VALID
------------------------------------------------------------
{'deliveryType': 'home', 'storeId': 'STORE-0042'}
INVALID: 2 error(s)
 1. $ [not] {'deliveryType': 'home', 'storeId': 'STORE-0042'} should not be valid under {'required': ['storeId']}
 2. $ [required] 'address' is a required property
------------------------------------------------------------
{'address': '10 Main Street', 'deliveryType': 'pickup'}
INVALID: 2 error(s)
 1. $ [not] {'deliveryType': 'pickup', 'address': '10 Main Street'} should not be valid under {'required': ['address']}
 2. $ [required] 'storeId' is a required property
------------------------------------------------------------


## Problem 7 Summary

Conditional schemas are useful when:

- one property changes the requirements for other properties,
- the document still conceptually represents one object type,
- the rules are easier to understand as a condition than as many `oneOf` branches.


---

# Problem 8 — Field Dependencies with `dependentRequired`

Suppose a customer may enable SMS alerts.

The JSON object contains:

- `email`,
- optional `phone`,
- optional `smsAlerts`.

If `smsAlerts` is present, then `phone` must also be present.

This is a dependency between properties.


## Step 1 — Define the Properties


In [40]:
account_preferences_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "email": {
            "type": "string",
            "format": "email",
        },
        "phone": {
            "type": "string",
            "pattern": "^\\+[1-9][0-9]{7,14}$",
        },
        "smsAlerts": {
            "type": "boolean",
        },
    },
    "required": ["email"],
    "additionalProperties": False,
}

## Step 2 — Add the Dependency

`dependentRequired` maps one property to other properties that must appear with it.


In [41]:
ACCOUNT_PREFERENCES_SCHEMA = {
    **account_preferences_schema,
    "dependentRequired": {
        "smsAlerts": ["phone"],
    },
}

check_schema(ACCOUNT_PREFERENCES_SCHEMA)

Schema definition is valid.


In [42]:
preference_cases = [
    {
        "email": "a@example.com",
    },
    {
        "email": "a@example.com",
        "smsAlerts": True,
    },
    {
        "email": "a@example.com",
        "smsAlerts": False,
        "phone": "+359888123456",
    },
]

for case in preference_cases:
    print(case)
    print_errors(case, ACCOUNT_PREFERENCES_SCHEMA)
    print()

{'email': 'a@example.com'}
VALID

{'email': 'a@example.com', 'smsAlerts': True}
INVALID: 1 error(s)
 1. $ [dependentRequired] 'phone' is a dependency of 'smsAlerts'

{'email': 'a@example.com', 'smsAlerts': False, 'phone': '+359888123456'}
VALID



Notice something subtle:

`dependentRequired` is about **presence**, not about the value of `smsAlerts`.

So even this:

```json
"smsAlerts": false
```

still triggers the requirement for `phone` if `smsAlerts` is present.

If we wanted the dependency only when the value is `true`, a conditional `if` / `then` rule would be more appropriate.


---

# Problem 9 — Tuple-Like Arrays with `prefixItems`

JSON arrays are usually collections of similar values.

But sometimes an array is used like a fixed-position tuple.

Suppose a sensor sample is stored as:

```text
[timestamp, temperature, unit]
```

For example:

```json
["2026-08-07T10:15:00Z", 23.4, "C"]
```

Each position has a different meaning.


## Step 1 — Why `items` Alone Is Not Enough

If we wrote:

```json
"items": {"type": "number"}
```

then every element would need to be a number.

That is not what we want.

Draft 2020-12 provides `prefixItems` for position-specific array schemas.


## Step 2 — Build the Tuple Schema


In [43]:
SENSOR_SAMPLE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "prefixItems": [
        {
            "type": "string",
            "format": "date-time",
        },
        {
            "type": "number",
            "minimum": -100,
            "maximum": 200,
        },
        {
            "enum": ["C", "F"],
        },
    ],
    "items": False,
    "minItems": 3,
    "maxItems": 3,
}

check_schema(SENSOR_SAMPLE_SCHEMA)

Schema definition is valid.


The important combination is:

- `prefixItems` describes positions,
- `items: false` blocks additional positions,
- `minItems` and `maxItems` make the length exactly 3.


In [44]:
samples = [
    ["2026-08-07T10:15:00Z", 23.4, "C"],
    ["not-a-date", 23.4, "C"],
    ["2026-08-07T10:15:00Z", 500, "C"],
    ["2026-08-07T10:15:00Z", 23.4, "K"],
    ["2026-08-07T10:15:00Z", 23.4, "C", "extra"],
]

for sample in samples:
    print(sample)
    print_errors(sample, SENSOR_SAMPLE_SCHEMA)
    print()

['2026-08-07T10:15:00Z', 23.4, 'C']
VALID

['not-a-date', 23.4, 'C']
VALID

['2026-08-07T10:15:00Z', 500, 'C']
INVALID: 1 error(s)
 1. $[1] [maximum] 500 is greater than the maximum of 200

['2026-08-07T10:15:00Z', 23.4, 'K']
INVALID: 1 error(s)
 1. $[2] [enum] 'K' is not one of ['C', 'F']

['2026-08-07T10:15:00Z', 23.4, 'C', 'extra']
INVALID: 2 error(s)
 1. $ [items] Expected at most 3 items but found 1 extra: 'extra'
 2. $ [maxItems] ['2026-08-07T10:15:00Z', 23.4, 'C', 'extra'] is too long



---

# Problem 10 — Requiring Matching Array Elements with `contains`

Suppose a deployment plan is an array of services.

Each service has a role:

- `frontend`,
- `backend`,
- `database`,
- `worker`.

A production deployment must contain **at least two database services**.

We do not care where they appear in the array.


## Step 1 — Define a Service


In [45]:
service_schema = {
    "type": "object",
    "properties": {
        "name": {
            "type": "string",
            "pattern": "^[a-z][a-z0-9-]{2,30}$",
        },
        "role": {
            "enum": [
                "frontend",
                "backend",
                "database",
                "worker",
            ],
        },
    },
    "required": ["name", "role"],
    "additionalProperties": False,
}

## Step 2 — Describe the Full Array


In [46]:
deployment_schema_step_2 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "items": service_schema,
}

check_schema(deployment_schema_step_2)

Schema definition is valid.


## Step 3 — Use `contains`

The `contains` keyword asks:

> Does this array contain an element that matches this schema?


In [47]:
database_matcher = {
    "type": "object",
    "properties": {
        "role": {
            "const": "database"
        }
    },
    "required": ["role"],
}

## Step 4 — Require At Least Two Matches

Draft 2020-12 supports `minContains`.


In [48]:
DEPLOYMENT_SCHEMA = {
    **deployment_schema_step_2,
    "contains": database_matcher,
    "minContains": 2,
}

check_schema(DEPLOYMENT_SCHEMA)

Schema definition is valid.


In [49]:
deployment_1 = [
    {"name": "web-01", "role": "frontend"},
    {"name": "api-01", "role": "backend"},
    {"name": "db-01", "role": "database"},
]

deployment_2 = [
    {"name": "web-01", "role": "frontend"},
    {"name": "db-01", "role": "database"},
    {"name": "db-02", "role": "database"},
]

print("Deployment 1")
print_errors(deployment_1, DEPLOYMENT_SCHEMA)

print("\nDeployment 2")
print_errors(deployment_2, DEPLOYMENT_SCHEMA)

Deployment 1
INVALID: 1 error(s)
 1. $ [minContains] Too few items match the given schema (expected at least 2 but only 1 matched)

Deployment 2
VALID


## Step 5 — A Rule JSON Schema Does Not Express Easily

Suppose service names must also be unique.

`uniqueItems: true` would only ensure that complete objects are not duplicated.

It would not directly mean:

> the `name` field must be unique across all objects.

That kind of property-based uniqueness is often clearer in Python.


In [50]:
def duplicate_service_names(services):
    seen = set()
    duplicates = set()

    for service in services:
        name = service.get("name")

        if name in seen:
            duplicates.add(name)

        seen.add(name)

    return sorted(duplicates)


deployment_with_duplicate_name = [
    {"name": "db-01", "role": "database"},
    {"name": "db-01", "role": "database"},
]

print(
    "Schema valid:",
    is_valid(
        deployment_with_duplicate_name,
        DEPLOYMENT_SCHEMA,
    ),
)

print(
    "Duplicate names:",
    duplicate_service_names(
        deployment_with_duplicate_name
    ),
)

Schema valid: True
Duplicate names: ['db-01']


---

# Problem 11 — Dynamic Object Keys with `patternProperties`

Suppose we store application metrics in a JSON object:

```json
{
    "cpu.user": 10.5,
    "cpu.system": 3.2,
    "requests.total": 1500
}
```

The property names are not known in advance.

So ordinary `properties` is not the right tool.


## Step 1 — Define the Key Pattern

Metric names may contain:

- lowercase letters,
- digits,
- `_`,
- `.`

They must begin with a lowercase letter or digit.


In [51]:
metric_name_pattern = "^[a-z0-9][a-z0-9_.]*$"

metric_name_pattern

'^[a-z0-9][a-z0-9_.]*$'

## Step 2 — Validate Property Names

`propertyNames` validates the names of keys.


In [52]:
metric_property_names_schema = {
    "type": "object",
    "propertyNames": {
        "pattern": metric_name_pattern
    },
}

for metrics in [
    {"cpu.user": 10},
    {"CPU.User": 10},
]:
    print(metrics)
    print_errors(metrics, metric_property_names_schema)
    print()

{'cpu.user': 10}
VALID

{'CPU.User': 10}
INVALID: 1 error(s)
 1. $ [pattern] 'CPU.User' does not match '^[a-z0-9][a-z0-9_.]*$'



## Step 3 — Validate the Values of Matching Keys

`patternProperties` associates a regular expression for key names with a schema for their values.


In [53]:
METRICS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "minProperties": 1,
    "propertyNames": {
        "pattern": metric_name_pattern,
    },
    "patternProperties": {
        metric_name_pattern: {
            "type": "number",
            "minimum": 0,
        }
    },
    "additionalProperties": False,
}

check_schema(METRICS_SCHEMA)

Schema definition is valid.


In [54]:
metrics_cases = [
    {
        "cpu.user": 10.5,
        "cpu.system": 3.2,
        "requests.total": 1500,
    },
    {
        "CPU.user": 10.5,
        "requests.total": -1,
    },
]

for case in metrics_cases:
    pprint(case)
    print_errors(case, METRICS_SCHEMA)
    print("-" * 60)

{'cpu.system': 3.2, 'cpu.user': 10.5, 'requests.total': 1500}
VALID
------------------------------------------------------------
{'CPU.user': 10.5, 'requests.total': -1}
INVALID: 3 error(s)
 1. $ [additionalProperties] 'CPU.user' does not match any of the regexes: '^[a-z0-9][a-z0-9_.]*$'
 2. $ [pattern] 'CPU.user' does not match '^[a-z0-9][a-z0-9_.]*$'
 3. $.requests.total [minimum] -1 is less than the minimum of 0
------------------------------------------------------------


---

# Problem 12 — Combining Schemas with `allOf`

Suppose we have a generic API entity:

```json
{
    "id": "ENT-123456",
    "createdAt": "2026-08-07T10:00:00Z"
}
```

We want to build a specialized product entity that also requires:

- `name`,
- `price`.

This is an example of composition.


## Step 1 — Base Entity


In [55]:
base_entity_schema = {
    "type": "object",
    "properties": {
        "id": {
            "type": "string",
            "pattern": "^ENT-[0-9]{6}$",
        },
        "createdAt": {
            "type": "string",
            "format": "date-time",
        },
    },
    "required": ["id", "createdAt"],
}

## Step 2 — Product-Specific Part


In [56]:
product_extension_schema = {
    "type": "object",
    "properties": {
        "name": {
            "type": "string",
            "minLength": 1,
        },
        "price": {
            "type": "number",
            "minimum": 0,
        },
    },
    "required": ["name", "price"],
}

## Step 3 — Compose with `allOf`

`allOf` means the instance must satisfy **every** listed schema.


In [57]:
PRODUCT_ENTITY_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "allOf": [
        base_entity_schema,
        product_extension_schema,
    ],
    "unevaluatedProperties": False,
}

check_schema(PRODUCT_ENTITY_SCHEMA)

Schema definition is valid.


Why use `unevaluatedProperties: false` here instead of placing `additionalProperties: false` inside each branch?

Because closed object schemas combined through `allOf` can reject properties introduced by neighboring branches.

`unevaluatedProperties` is designed to close the composed object after all branches have had a chance to evaluate their fields.


In [58]:
product_entity = {
    "id": "ENT-123456",
    "createdAt": "2026-08-07T10:00:00Z",
    "name": "Mechanical Keyboard",
    "price": 89.99,
}

print_errors(
    product_entity,
    PRODUCT_ENTITY_SCHEMA,
)

VALID


In [59]:
broken_product_entity = {
    "id": "ENT-12",
    "createdAt": "yesterday",
    "name": "",
    "price": -10,
    "debug": True,
}

print_errors(
    broken_product_entity,
    PRODUCT_ENTITY_SCHEMA,
)

INVALID: 4 error(s)
 1. $ [unevaluatedProperties] Unevaluated properties are not allowed ('createdAt', 'debug', 'id', 'name', 'price' were unexpected)
 2. $.id [pattern] 'ENT-12' does not match '^ENT-[0-9]{6}$'
 3. $.name [minLength] '' should be non-empty
 4. $.price [minimum] -10 is less than the minimum of 0


---

# Problem 13 — Recursive Schemas

Some JSON structures can contain smaller versions of themselves.

Examples:

- directory trees,
- menus,
- expression trees,
- organization charts,
- nested comments.

We will model a nested menu.


A menu item looks like:

```json
{
    "label": "Products",
    "children": [
        {
            "label": "Books"
        },
        {
            "label": "Software"
        }
    ]
}
```

Every child is itself another menu item.


## Step 1 — Define One Node

Each node requires:

- `label`,
- optional `url`,
- optional `children`.


## Step 2 — Make `children` Reference the Node Definition

The key idea is:

```json
"items": {
    "$ref": "#/$defs/menuItem"
}
```

The definition points back to itself.


In [60]:
MENU_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "menuItem": {
            "type": "object",
            "properties": {
                "label": {
                    "type": "string",
                    "minLength": 1,
                },
                "url": {
                    "type": "string",
                    "format": "uri-reference",
                },
                "children": {
                    "type": "array",
                    "items": {
                        "$ref": "#/$defs/menuItem"
                    },
                },
            },
            "required": ["label"],
            "additionalProperties": False,
        }
    },
    "$ref": "#/$defs/menuItem",
}

check_schema(MENU_SCHEMA)

Schema definition is valid.


## Step 3 — Validate a Deep Structure


In [61]:
menu = {
    "label": "Root",
    "children": [
        {
            "label": "Products",
            "children": [
                {
                    "label": "Books",
                    "url": "/products/books",
                },
                {
                    "label": "Software",
                    "children": [
                        {
                            "label": "Developer Tools",
                            "url": "/products/software/dev",
                        }
                    ],
                },
            ],
        }
    ],
}

print_errors(menu, MENU_SCHEMA)

VALID


## Step 4 — Introduce a Deep Error

Notice how the error path allows us to find the exact broken descendant.


In [62]:
broken_menu = {
    "label": "Root",
    "children": [
        {
            "label": "Products",
            "children": [
                {
                    "label": "",
                    "unexpected": 123,
                }
            ],
        }
    ],
}

print_errors(broken_menu, MENU_SCHEMA)

INVALID: 2 error(s)
 1. $.children[0].children[0] [additionalProperties] Additional properties are not allowed ('unexpected' was unexpected)
 2. $.children[0].children[0].label [minLength] '' should be non-empty


---

# Problem 14 — Versioned API Documents

APIs evolve.

Suppose version 1 of a search request uses:

```json
{
    "version": 1,
    "query": "python",
    "limit": 10
}
```

Version 2 uses:

```json
{
    "version": 2,
    "query": "python",
    "pageSize": 10,
    "includeArchived": false
}
```

The field names changed between versions.

We want one schema that accepts either version.


## Step 1 — Define Version 1

The `version` value itself acts as the discriminator.


In [63]:
search_v1_schema = {
    "type": "object",
    "properties": {
        "version": {
            "const": 1
        },
        "query": {
            "type": "string",
            "minLength": 1,
        },
        "limit": {
            "type": "integer",
            "minimum": 1,
            "maximum": 100,
        },
    },
    "required": [
        "version",
        "query",
        "limit",
    ],
    "additionalProperties": False,
}

## Step 2 — Define Version 2


In [64]:
search_v2_schema = {
    "type": "object",
    "properties": {
        "version": {
            "const": 2
        },
        "query": {
            "type": "string",
            "minLength": 1,
        },
        "pageSize": {
            "type": "integer",
            "minimum": 1,
            "maximum": 200,
        },
        "includeArchived": {
            "type": "boolean",
        },
    },
    "required": [
        "version",
        "query",
        "pageSize",
        "includeArchived",
    ],
    "additionalProperties": False,
}

## Step 3 — Combine the Versions


In [65]:
SEARCH_REQUEST_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "oneOf": [
        search_v1_schema,
        search_v2_schema,
    ],
}

check_schema(SEARCH_REQUEST_SCHEMA)

Schema definition is valid.


## Step 4 — Test Both Versions


In [66]:
search_requests = [
    {
        "version": 1,
        "query": "python",
        "limit": 10,
    },
    {
        "version": 2,
        "query": "python",
        "pageSize": 20,
        "includeArchived": False,
    },
    {
        "version": 2,
        "query": "python",
        "limit": 10,
    },
]

for request in search_requests:
    pprint(request)
    print_errors(request, SEARCH_REQUEST_SCHEMA)
    print("-" * 60)

{'limit': 10, 'query': 'python', 'version': 1}
VALID
------------------------------------------------------------
{'includeArchived': False, 'pageSize': 20, 'query': 'python', 'version': 2}
VALID
------------------------------------------------------------
{'limit': 10, 'query': 'python', 'version': 2}
INVALID: 1 error(s)
 1. $ [oneOf] {'version': 2, 'query': 'python', 'limit': 10} is not valid under any of the given schemas
------------------------------------------------------------


## Step 5 — Migration After Validation

Schema validation tells us whether the document matches a supported version.

After that, application code can normalize older versions.


In [67]:
def migrate_search_request_to_v2(request):
    errors = collect_errors(
        request,
        SEARCH_REQUEST_SCHEMA,
    )

    if errors:
        raise ValueError(
            "Request must be valid before migration."
        )

    if request["version"] == 2:
        return dict(request)

    return {
        "version": 2,
        "query": request["query"],
        "pageSize": request["limit"],
        "includeArchived": False,
    }


old_request = {
    "version": 1,
    "query": "json schema",
    "limit": 25,
}

pprint(
    migrate_search_request_to_v2(
        old_request
    )
)

{'includeArchived': False, 'pageSize': 25, 'query': 'json schema', 'version': 2}


---

# Problem 15 — JSON Text Errors vs Schema Errors

A very common confusion is to treat malformed JSON and schema-invalid JSON as the same problem.

They are different stages.

Consider this string:


In [68]:
malformed_json = '''
{
    "studentId": "STU-20260001",
    "firstName": "Ada",
}
'''

The trailing comma is illegal JSON syntax.

The schema validator cannot validate this text directly because we do not yet have a Python object.


In [69]:
try:
    parsed = json.loads(malformed_json)
except json.JSONDecodeError as exc:
    print("JSON syntax error")
    print("Line:", exc.lineno)
    print("Column:", exc.colno)
    print("Message:", exc.msg)

JSON syntax error
Line: 4
Column: 23
Message: Illegal trailing comma before end of object


Now compare that with syntactically valid JSON whose values violate the schema.


In [70]:
schema_invalid_json = '''
{
    "studentId": "BAD-ID",
    "firstName": "",
    "lastName": "Lovelace",
    "year": 10,
    "status": "unknown"
}
'''

parsed = json.loads(schema_invalid_json)

print_errors(
    parsed,
    STUDENT_SCHEMA,
)

INVALID: 4 error(s)
 1. $.firstName [minLength] '' should be non-empty
 2. $.status [enum] 'unknown' is not one of ['active', 'suspended', 'graduated']
 3. $.studentId [pattern] 'BAD-ID' does not match '^STU-[0-9]{8}$'
 4. $.year [maximum] 10 is greater than the maximum of 6


## Step 3 — Build a Reusable Two-Stage Validator

A useful API helper can distinguish:

- JSON parsing problems,
- schema validation problems.


In [71]:
def validate_json_text(text, schema):
    try:
        instance = json.loads(text)
    except json.JSONDecodeError as exc:
        return {
            "ok": False,
            "kind": "json_syntax",
            "errors": [
                {
                    "path": "$",
                    "line": exc.lineno,
                    "column": exc.colno,
                    "message": exc.msg,
                }
            ],
        }

    errors = collect_errors(
        instance,
        schema,
    )

    if errors:
        return {
            "ok": False,
            "kind": "schema_validation",
            "errors": [
                {
                    "path": path_to_string(
                        error.absolute_path
                    ),
                    "validator": error.validator,
                    "message": error.message,
                }
                for error in errors
            ],
        }

    return {
        "ok": True,
        "kind": "valid",
        "value": instance,
        "errors": [],
    }

In [72]:
for document in [
    malformed_json,
    schema_invalid_json,
    json.dumps(student_1),
]:
    pprint(
        validate_json_text(
            document,
            STUDENT_SCHEMA,
        )
    )
    print("=" * 80)

{'errors': [{'column': 23,
             'line': 4,
             'message': 'Illegal trailing comma before end of object',
             'path': '$'}],
 'kind': 'json_syntax',
 'ok': False}
{'errors': [{'message': "'' should be non-empty",
             'path': '$.firstName',
             'validator': 'minLength'},
            {'message': "'unknown' is not one of ['active', 'suspended', "
                        "'graduated']",
             'path': '$.status',
             'validator': 'enum'},
            {'message': "'BAD-ID' does not match '^STU-[0-9]{8}$'",
             'path': '$.studentId',
             'validator': 'pattern'},
            {'message': '10 is greater than the maximum of 6',
             'path': '$.year',
             'validator': 'maximum'}],
 'kind': 'schema_validation',
 'ok': False}
{'errors': [],
 'kind': 'valid',
 'ok': True,
 'value': {'firstName': 'Ada',
           'gpa': 3.92,
           'lastName': 'Lovelace',
           'status': 'active',
           'stude

---

# Problem 16 — Group Validation Errors by Path

When returning errors from a REST API, a flat sequence can be useful.

Sometimes it is more convenient for a frontend to receive errors grouped by field.


For example:

```python
{
    "$.year": [
        "10 is greater than the maximum of 6"
    ],
    "$.status": [
        "'unknown' is not one of ..."
    ]
}
```


In [73]:
def errors_by_path(instance, schema):
    grouped = defaultdict(list)

    for error in collect_errors(
        instance,
        schema,
    ):
        path = path_to_string(
            error.absolute_path
        )

        grouped[path].append(
            error.message
        )

    return dict(grouped)

In [74]:
bad_student = {
    "studentId": "BAD",
    "firstName": "",
    "lastName": "",
    "year": 10,
    "gpa": -1,
    "status": "unknown",
    "debug": True,
}

pprint(
    errors_by_path(
        bad_student,
        STUDENT_SCHEMA,
    )
)

{'$': ["Additional properties are not allowed ('debug' was unexpected)"],
 '$.firstName': ["'' should be non-empty"],
 '$.gpa': ['-1 is less than the minimum of 0'],
 '$.lastName': ["'' should be non-empty"],
 '$.status': ["'unknown' is not one of ['active', 'suspended', 'graduated']"],
 '$.studentId': ["'BAD' does not match '^STU-[0-9]{8}$'"],
 '$.year': ['10 is greater than the maximum of 6']}


This is not part of JSON Schema itself.

It is application-level presentation logic built on top of the validator's error objects.


---

# Problem 17 — Validating a Batch of Imported Records

Imagine importing thousands of student records from another system.

We want to:

- count valid records,
- count invalid records,
- preserve the index of each failed record,
- store compact validation errors.


## Step 1 — Design the Report Shape

We will return:

```python
{
    "total": ...,
    "valid": ...,
    "invalid": ...,
    "failures": [...]
}
```


In [75]:
def validate_batch(records, schema):
    failures = []

    for index, record in enumerate(records):
        errors = collect_errors(
            record,
            schema,
        )

        if not errors:
            continue

        failures.append({
            "index": index,
            "errors": [
                {
                    "path": path_to_string(
                        error.absolute_path
                    ),
                    "message": error.message,
                }
                for error in errors
            ],
        })

    return {
        "total": len(records),
        "valid": len(records) - len(failures),
        "invalid": len(failures),
        "failures": failures,
    }

## Step 2 — Run the Batch


In [76]:
student_batch = [
    {
        "studentId": "STU-20260001",
        "firstName": "Ada",
        "lastName": "Lovelace",
        "year": 2,
        "status": "active",
    },
    {
        "studentId": "BAD",
        "firstName": "Alan",
        "lastName": "Turing",
        "year": 2,
        "status": "active",
    },
    {
        "studentId": "STU-20260003",
        "firstName": "",
        "lastName": "Hopper",
        "year": 3,
        "status": "active",
    },
]

report = validate_batch(
    student_batch,
    STUDENT_SCHEMA,
)

pprint(report)

{'failures': [{'errors': [{'message': "'BAD' does not match '^STU-[0-9]{8}$'",
                           'path': '$.studentId'}],
               'index': 1},
              {'errors': [{'message': "'' should be non-empty",
                           'path': '$.firstName'}],
               'index': 2}],
 'invalid': 2,
 'total': 3,
 'valid': 1}


---

# Problem 18 — Schema Tests as Executable Documentation

A schema is part of an application contract.

That means it should be tested.

We can treat examples as executable documentation:

- known-good examples should remain valid,
- known-bad examples should remain invalid.


## Step 1 — Build Assertion Helpers


In [77]:
def assert_valid(instance, schema):
    errors = collect_errors(
        instance,
        schema,
    )

    if errors:
        detail = "\n".join(
            f"{path_to_string(error.absolute_path)}: "
            f"{error.message}"
            for error in errors
        )

        raise AssertionError(
            "Expected valid instance, but found:\n"
            + detail
        )


def assert_invalid(instance, schema):
    errors = collect_errors(
        instance,
        schema,
    )

    if not errors:
        raise AssertionError(
            "Expected invalid instance, "
            "but validation succeeded."
        )

## Step 2 — Test the Schema Definition Itself

This catches mistakes such as invalid keyword values.


In [78]:
Draft202012Validator.check_schema(
    STUDENT_SCHEMA
)

print("Student schema definition is valid.")

Student schema definition is valid.


## Step 3 — Add Regression Cases


In [79]:
assert_valid(
    {
        "studentId": "STU-20260001",
        "firstName": "Ada",
        "lastName": "Lovelace",
        "year": 2,
        "status": "active",
    },
    STUDENT_SCHEMA,
)

assert_invalid(
    {
        "studentId": "STU-20260001",
        "firstName": "Ada",
        "lastName": "Lovelace",
        "year": 0,
        "status": "active",
    },
    STUDENT_SCHEMA,
)

assert_invalid(
    {
        "studentId": "STU-20260001",
        "firstName": "Ada",
        "lastName": "Lovelace",
        "year": 2,
        "status": "active",
        "extra": True,
    },
    STUDENT_SCHEMA,
)

print("All tests passed.")

All tests passed.


## Step 4 — Table-Driven Tests

A table makes it easy to add many examples without repeating assertion code.


In [80]:
student_test_cases = [
    (
        "minimal valid",
        {
            "studentId": "STU-20260001",
            "firstName": "Ada",
            "lastName": "Lovelace",
            "year": 2,
            "status": "active",
        },
        True,
    ),
    (
        "bad id",
        {
            "studentId": "123",
            "firstName": "Ada",
            "lastName": "Lovelace",
            "year": 2,
            "status": "active",
        },
        False,
    ),
    (
        "gpa optional",
        {
            "studentId": "STU-20260001",
            "firstName": "Ada",
            "lastName": "Lovelace",
            "year": 2,
            "status": "active",
        },
        True,
    ),
    (
        "gpa too high",
        {
            "studentId": "STU-20260001",
            "firstName": "Ada",
            "lastName": "Lovelace",
            "year": 2,
            "gpa": 5,
            "status": "active",
        },
        False,
    ),
]

for name, instance, expected in student_test_cases:
    actual = is_valid(
        instance,
        STUDENT_SCHEMA,
    )

    result = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    print(
        f"{result:4} | "
        f"{name:18} | "
        f"expected={expected} "
        f"actual={actual}"
    )

PASS | minimal valid      | expected=True actual=True
PASS | bad id             | expected=False actual=False
PASS | gpa optional       | expected=True actual=True
PASS | gpa too high       | expected=False actual=False


---

# Problem 19 — A Business Rule That Belongs in Python

JSON Schema is excellent for structural validation.

But not every rule is naturally structural.

Suppose an invoice line contains:

```json
{
    "quantity": 3,
    "unitPrice": 19.95,
    "lineTotal": 59.85
}
```

We want:

```text
lineTotal = quantity × unitPrice
```

This arithmetic relationship is clearer to enforce in Python.


## Step 1 — Use JSON Schema for Shape and Ranges


In [81]:
INVOICE_LINE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "quantity": {
            "type": "integer",
            "minimum": 1,
        },
        "unitPrice": {
            "type": "number",
            "minimum": 0,
        },
        "lineTotal": {
            "type": "number",
            "minimum": 0,
        },
    },
    "required": [
        "quantity",
        "unitPrice",
        "lineTotal",
    ],
    "additionalProperties": False,
}

check_schema(INVOICE_LINE_SCHEMA)

Schema definition is valid.


## Step 2 — Add Business Validation


In [82]:
def validate_invoice_line(line):
    errors = collect_errors(
        line,
        INVOICE_LINE_SCHEMA,
    )

    if errors:
        return [
            {
                "path": path_to_string(
                    error.absolute_path
                ),
                "message": error.message,
            }
            for error in errors
        ]

    expected = round(
        line["quantity"]
        * line["unitPrice"],
        2,
    )

    actual = round(
        line["lineTotal"],
        2,
    )

    if expected != actual:
        return [
            {
                "path": "$.lineTotal",
                "message": (
                    f"Expected {expected}, "
                    f"got {actual}"
                ),
            }
        ]

    return []

In [83]:
invoice_lines = [
    {
        "quantity": 3,
        "unitPrice": 19.95,
        "lineTotal": 59.85,
    },
    {
        "quantity": 3,
        "unitPrice": 19.95,
        "lineTotal": 60.00,
    },
    {
        "quantity": 0,
        "unitPrice": 19.95,
        "lineTotal": 0,
    },
]

for line in invoice_lines:
    print(line)
    pprint(validate_invoice_line(line))
    print()

{'quantity': 3, 'unitPrice': 19.95, 'lineTotal': 59.85}
[]

{'quantity': 3, 'unitPrice': 19.95, 'lineTotal': 60.0}
[{'message': 'Expected 59.85, got 60.0', 'path': '$.lineTotal'}]

{'quantity': 0, 'unitPrice': 19.95, 'lineTotal': 0}
[{'message': '0 is less than the minimum of 1', 'path': '$.quantity'}]



## Problem 19 Summary

A useful boundary is:

### JSON Schema
Use it for:

- shape,
- allowed fields,
- primitive types,
- ranges,
- patterns,
- alternatives,
- dependencies,
- nested structures.

### Application code
Use it for rules involving:

- calculations,
- database lookups,
- uniqueness against external data,
- permissions,
- time-dependent business state,
- relationships that are clearer in code.


---

# Problem 20 — Capstone: Job Submission API

We will now combine many ideas into one realistic schema.

Imagine a system where users submit jobs to a processing platform.

A request looks roughly like this:

```json
{
    "requestId": "...uuid...",
    "submittedAt": "...date-time...",
    "owner": {...},
    "job": {...},
    "notifications": {...}
}
```

We will build the schema in logical layers.


## Capstone Step 1 — Request Metadata

Every request requires:

- `requestId`: UUID,
- `submittedAt`: date-time.


In [84]:
request_metadata_properties = {
    "requestId": {
        "type": "string",
        "format": "uuid",
    },
    "submittedAt": {
        "type": "string",
        "format": "date-time",
    },
}

## Capstone Step 2 — Owner

The owner object requires:

- `userId`,
- `email`.

The object is strict.


In [85]:
owner_schema = {
    "type": "object",
    "properties": {
        "userId": {
            "type": "string",
            "pattern": "^USR-[0-9]{8}$",
        },
        "email": {
            "type": "string",
            "format": "email",
        },
    },
    "required": [
        "userId",
        "email",
    ],
    "additionalProperties": False,
}

## Capstone Step 3 — Job Variants

The platform supports three job types:

1. `report`,
2. `export`,
3. `cleanup`.

Each job type has different fields.

This strongly suggests `oneOf`.


### Report Job

A report job requires:

- `type = "report"`,
- `reportName`,
- `format` of `pdf` or `html`.


In [86]:
report_job_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "report",
        },
        "reportName": {
            "type": "string",
            "minLength": 1,
        },
        "format": {
            "enum": ["pdf", "html"],
        },
    },
    "required": [
        "type",
        "reportName",
        "format",
    ],
    "additionalProperties": False,
}

### Export Job

An export job requires:

- `type = "export"`,
- `resource`,
- `format` of `csv` or `json`,
- optional `delimiter`.

But `delimiter` should only be allowed for CSV.


We can solve that with a small conditional inside the export branch.


In [87]:
export_job_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "export",
        },
        "resource": {
            "type": "string",
            "minLength": 1,
        },
        "format": {
            "enum": ["csv", "json"],
        },
        "delimiter": {
            "type": "string",
            "minLength": 1,
            "maxLength": 1,
        },
    },
    "required": [
        "type",
        "resource",
        "format",
    ],
    "if": {
        "properties": {
            "format": {
                "const": "csv"
            }
        },
        "required": ["format"],
    },
    "then": {},
    "else": {
        "not": {
            "required": ["delimiter"]
        }
    },
    "additionalProperties": False,
}

### Cleanup Job

A cleanup job requires:

- `type = "cleanup"`,
- `olderThanDays`: integer from 1 to 3650,
- `dryRun`: boolean.


In [88]:
cleanup_job_schema = {
    "type": "object",
    "properties": {
        "type": {
            "const": "cleanup",
        },
        "olderThanDays": {
            "type": "integer",
            "minimum": 1,
            "maximum": 3650,
        },
        "dryRun": {
            "type": "boolean",
        },
    },
    "required": [
        "type",
        "olderThanDays",
        "dryRun",
    ],
    "additionalProperties": False,
}

## Capstone Step 4 — Combine the Job Variants


In [89]:
job_schema = {
    "oneOf": [
        report_job_schema,
        export_job_schema,
        cleanup_job_schema,
    ],
}

## Capstone Step 5 — Notification Preferences

Notifications may specify any combination of:

- email,
- SMS.

But at least one destination must exist.


In [90]:
notifications_schema = {
    "type": "object",
    "properties": {
        "email": {
            "type": "string",
            "format": "email",
        },
        "sms": {
            "type": "string",
            "pattern": "^\\+[1-9][0-9]{7,14}$",
        },
    },
    "anyOf": [
        {"required": ["email"]},
        {"required": ["sms"]},
    ],
    "additionalProperties": False,
}

## Capstone Step 6 — Assemble the Full Schema


In [91]:
JOB_SUBMISSION_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "owner": owner_schema,
        "job": job_schema,
        "notifications": notifications_schema,
    },
    "type": "object",
    "properties": {
        **request_metadata_properties,
        "owner": {
            "$ref": "#/$defs/owner"
        },
        "job": {
            "$ref": "#/$defs/job"
        },
        "notifications": {
            "$ref": "#/$defs/notifications"
        },
    },
    "required": [
        "requestId",
        "submittedAt",
        "owner",
        "job",
        "notifications",
    ],
    "additionalProperties": False,
}

check_schema(JOB_SUBMISSION_SCHEMA)

Schema definition is valid.


## Capstone Step 7 — Validate a Report Job


In [92]:
report_submission = {
    "requestId": "f4b79a2b-c06a-4a1b-b438-303f0069ecae",
    "submittedAt": "2026-08-07T12:00:00Z",
    "owner": {
        "userId": "USR-00001234",
        "email": "ada@example.com",
    },
    "job": {
        "type": "report",
        "reportName": "Monthly Revenue",
        "format": "pdf",
    },
    "notifications": {
        "email": "ada@example.com",
    },
}

print_errors(
    report_submission,
    JOB_SUBMISSION_SCHEMA,
)

VALID


## Capstone Step 8 — Validate an Export Job


In [93]:
export_submission = {
    "requestId": "20e2907f-dc96-462b-9251-5727e37d5c53",
    "submittedAt": "2026-08-07T12:05:00Z",
    "owner": {
        "userId": "USR-00005678",
        "email": "grace@example.com",
    },
    "job": {
        "type": "export",
        "resource": "orders",
        "format": "csv",
        "delimiter": ";",
    },
    "notifications": {
        "email": "grace@example.com",
        "sms": "+359888123456",
    },
}

print_errors(
    export_submission,
    JOB_SUBMISSION_SCHEMA,
)

VALID


## Capstone Step 9 — A JSON Export Must Not Have a Delimiter


In [94]:
bad_json_export = {
    **export_submission,
    "job": {
        "type": "export",
        "resource": "orders",
        "format": "json",
        "delimiter": ",",
    },
}

print_errors(
    bad_json_export,
    JOB_SUBMISSION_SCHEMA,
)

INVALID: 1 error(s)
 1. $.job [oneOf] {'type': 'export', 'resource': 'orders', 'format': 'json', 'delimiter': ','} is not valid under any of the given schemas


## Capstone Step 10 — Inject Several Errors at Once

This is the kind of test that makes `iter_errors()` especially useful.


In [95]:
broken_submission = {
    "requestId": "not-a-uuid",
    "submittedAt": "yesterday",
    "owner": {
        "userId": "123",
        "email": "not-an-email",
    },
    "job": {
        "type": "cleanup",
        "olderThanDays": 0,
        "dryRun": "no",
    },
    "notifications": {},
    "unexpected": True,
}

print_errors(
    broken_submission,
    JOB_SUBMISSION_SCHEMA,
)

INVALID: 6 error(s)
 1. $ [additionalProperties] Additional properties are not allowed ('unexpected' was unexpected)
 2. $.job [oneOf] {'type': 'cleanup', 'olderThanDays': 0, 'dryRun': 'no'} is not valid under any of the given schemas
 3. $.notifications [anyOf] {} is not valid under any of the given schemas
 4. $.owner.email [format] 'not-an-email' is not a 'email'
 5. $.owner.userId [pattern] '123' does not match '^USR-[0-9]{8}$'
 6. $.requestId [format] 'not-a-uuid' is not a 'uuid'


## Capstone Step 11 — Convert Errors to API-Friendly Objects


In [96]:
def api_error_list(instance, schema):
    return [
        {
            "path": path_to_string(
                error.absolute_path
            ),
            "validator": error.validator,
            "message": error.message,
        }
        for error in collect_errors(
            instance,
            schema,
        )
    ]


pprint(
    api_error_list(
        broken_submission,
        JOB_SUBMISSION_SCHEMA,
    )
)

[{'message': "Additional properties are not allowed ('unexpected' was "
             'unexpected)',
  'path': '$',
  'validator': 'additionalProperties'},
 {'message': "{'type': 'cleanup', 'olderThanDays': 0, 'dryRun': 'no'} is not "
             'valid under any of the given schemas',
  'path': '$.job',
  'validator': 'oneOf'},
 {'message': '{} is not valid under any of the given schemas',
  'path': '$.notifications',
  'validator': 'anyOf'},
 {'message': "'not-an-email' is not a 'email'",
  'path': '$.owner.email',
  'validator': 'format'},
 {'message': "'123' does not match '^USR-[0-9]{8}$'",
  'path': '$.owner.userId',
  'validator': 'pattern'},
 {'message': "'not-a-uuid' is not a 'uuid'",
  'path': '$.requestId',
  'validator': 'format'}]


## Capstone Summary

The finished schema combines:

- primitive validation,
- `format`,
- strict objects,
- `$defs`,
- `$ref`,
- `oneOf`,
- `anyOf`,
- `if` / `then` / `else`,
- nested validation,
- reusable error reporting.

This is much closer to how JSON Schema is used in real API contracts.


---

# Additional Guided Problem A — Feature Rollout

A feature rollout document looks like:

```json
{
    "feature": "new-dashboard",
    "percentage": 25,
    "allowList": ["USR-00000001"]
}
```

Rules:

- `feature` required,
- `percentage` from 0 to 100,
- if percentage is less than 100, `allowList` is required and must contain at least one user ID,
- if percentage is 100, `allowList` is optional.


## Reasoning

The interesting rule depends on the numeric value of `percentage`.

This is a conditional validation problem.

We can:

1. define all fields,
2. require the common fields,
3. use `if` to detect percentages below 100,
4. require `allowList` in `then`.


In [97]:
FEATURE_ROLLOUT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "feature": {
            "type": "string",
            "minLength": 1,
        },
        "percentage": {
            "type": "number",
            "minimum": 0,
            "maximum": 100,
        },
        "allowList": {
            "type": "array",
            "minItems": 1,
            "uniqueItems": True,
            "items": {
                "type": "string",
                "pattern": "^USR-[0-9]{8}$",
            },
        },
    },
    "required": [
        "feature",
        "percentage",
    ],
    "if": {
        "properties": {
            "percentage": {
                "exclusiveMaximum": 100
            }
        },
        "required": ["percentage"],
    },
    "then": {
        "required": ["allowList"]
    },
    "additionalProperties": False,
}

check_schema(FEATURE_ROLLOUT_SCHEMA)

Schema definition is valid.


In [98]:
rollouts = [
    {
        "feature": "new-dashboard",
        "percentage": 100,
    },
    {
        "feature": "new-dashboard",
        "percentage": 25,
        "allowList": ["USR-00000001"],
    },
    {
        "feature": "new-dashboard",
        "percentage": 25,
    },
]

for rollout in rollouts:
    print(rollout)
    print_errors(
        rollout,
        FEATURE_ROLLOUT_SCHEMA,
    )
    print()

{'feature': 'new-dashboard', 'percentage': 100}
VALID

{'feature': 'new-dashboard', 'percentage': 25, 'allowList': ['USR-00000001']}
VALID

{'feature': 'new-dashboard', 'percentage': 25}
INVALID: 1 error(s)
 1. $ [required] 'allowList' is a required property



---

# Additional Guided Problem B — Matrix Validation

Represent a 2×3 matrix as:

```json
[
    [1, 2, 3],
    [4, 5, 6]
]
```

Requirements:

- exactly 2 rows,
- every row has exactly 3 values,
- every value is a number.


## Reasoning

This is an array containing arrays.

So the schema has two levels:

1. outer array constraints,
2. inner row constraints.


In [99]:
MATRIX_2X3_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "array",
    "minItems": 2,
    "maxItems": 2,
    "items": {
        "type": "array",
        "minItems": 3,
        "maxItems": 3,
        "items": {
            "type": "number"
        },
    },
}

check_schema(MATRIX_2X3_SCHEMA)

Schema definition is valid.


In [100]:
matrices = [
    [
        [1, 2, 3],
        [4, 5, 6],
    ],
    [
        [1, 2],
        [3, 4, 5],
    ],
    [
        [1, 2, 3],
        [4, "five", 6],
    ],
]

for matrix in matrices:
    pprint(matrix)
    print_errors(
        matrix,
        MATRIX_2X3_SCHEMA,
    )
    print("-" * 40)

[[1, 2, 3], [4, 5, 6]]
VALID
----------------------------------------
[[1, 2], [3, 4, 5]]
INVALID: 1 error(s)
 1. $[0] [minItems] [1, 2] is too short
----------------------------------------
[[1, 2, 3], [4, 'five', 6]]
INVALID: 1 error(s)
 1. $[1][1] [type] 'five' is not of type 'number'
----------------------------------------


---

# Additional Guided Problem C — Forbid a Dangerous Combination

A runtime configuration allows:

- `debug`,
- `production`,
- `cache`.

All are optional booleans.

But `debug: true` and `production: true` must never occur together.


## Reasoning

We can describe the forbidden pattern, then wrap it in `not`.

The forbidden pattern is an object where both properties exist and both are `true`.


In [101]:
RUNTIME_FLAGS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
        "debug": {
            "type": "boolean",
        },
        "production": {
            "type": "boolean",
        },
        "cache": {
            "type": "boolean",
        },
    },
    "not": {
        "properties": {
            "debug": {
                "const": True,
            },
            "production": {
                "const": True,
            },
        },
        "required": [
            "debug",
            "production",
        ],
    },
    "additionalProperties": False,
}

check_schema(RUNTIME_FLAGS_SCHEMA)

Schema definition is valid.


In [102]:
flag_sets = [
    {
        "debug": True,
        "production": False,
    },
    {
        "debug": False,
        "production": True,
    },
    {
        "debug": True,
        "production": True,
    },
]

for flags in flag_sets:
    print(flags)
    print_errors(
        flags,
        RUNTIME_FLAGS_SCHEMA,
    )
    print()

{'debug': True, 'production': False}
VALID

{'debug': False, 'production': True}
VALID

{'debug': True, 'production': True}
INVALID: 1 error(s)
 1. $ [not] {'debug': True, 'production': True} should not be valid under {'properties': {'debug': {'const': True}, 'production': {'const': True}}, 'required': ['debug', 'production']}



---

# Best Practices Review

After solving all of these problems, we can summarize several strong habits.


## 1. State the Draft Explicitly

Prefer:

```json
"$schema": "https://json-schema.org/draft/2020-12/schema"
```

This makes the intended keyword behavior explicit.


## 2. Validate Your Schemas

Use:

```python
Draft202012Validator.check_schema(schema)
```

A broken schema can otherwise lead to confusing behavior.


## 3. Build Incrementally

A reliable design process is:

1. outer type,
2. properties,
3. required fields,
4. primitive constraints,
5. nested structures,
6. cross-field rules,
7. strictness,
8. test cases.


## 4. Distinguish Optional from Nullable

These are different:

```json
"nickname": {"type": "string"}
```

and:

```json
"middleName": {"type": ["string", "null"]}
```

Whether the property is required is a separate decision.


## 5. Use Strict Objects Intentionally

For many API request bodies:

```json
"additionalProperties": false
```

helps catch typos and accidental fields.

However, do not use it blindly in schemas that are intentionally extensible.


## 6. Reuse Structures

Prefer `$defs` and `$ref` instead of duplicating long nested schemas.


## 7. Use Discriminators for Variants

A design such as:

```json
"type": {"const": "oauth"}
```

inside a `oneOf` branch makes variant schemas easier to reason about.


## 8. Remember That `format` Checking Is Validator-Dependent

When using Python's `jsonschema` package, pass a `FormatChecker` if you expect formats such as `email`, `uuid`, or `date-time` to be checked.


## 9. Prefer All Errors During Development

`iter_errors()` is often more useful than receiving only the first error.

It gives a much better debugging experience for large documents.


## 10. Keep Some Rules in Application Code

Not every business invariant belongs in JSON Schema.

Use ordinary Python when the rule depends on:

- arithmetic,
- external services,
- database state,
- authenticated user permissions,
- uniqueness against stored records,
- domain workflows.


---

# Final Independent Exercises

Try these without looking back at earlier solutions.

Each problem can be solved using techniques already demonstrated in this notebook.


## Exercise 1 — Pagination

Support exactly one pagination mode:

### Page mode

```json
{
    "mode": "page",
    "page": 3,
    "pageSize": 50
}
```

### Cursor mode

```json
{
    "mode": "cursor",
    "cursor": "abc123",
    "limit": 50
}
```

Use a discriminator and `oneOf`.


## Exercise 2 — Shipping Package

Create a strict package schema with:

- `weightKg`,
- `dimensions`,
- optional `insuredValue`,
- dimensions containing positive `widthCm`, `heightCm`, `depthCm`.


## Exercise 3 — Notification Event

Create a schema with variants:

- email notification,
- SMS notification,
- push notification.

Each variant should use a `type` discriminator and variant-specific fields.


## Exercise 4 — Search Filter Tree

Create a recursive schema where a node is either:

- a comparison leaf,
- an `and` node containing child filters,
- an `or` node containing child filters.

This combines recursion with `oneOf`.


## Exercise 5 — Import Pipeline

Validate a list of imported customer records and produce:

- total,
- valid,
- invalid,
- grouped errors by row,
- grouped errors by JSON path.


# End of Notebook

You have now worked through JSON Schema as a **design process**, not just as a list of keywords.

The most important skill is learning to translate a natural-language contract into small, testable schema rules and refine them one layer at a time.
